# ECDC Open Data Accessor - Example Notebook

This notebook demonstrates how to use the ECDC Open Data accessor to fetch epidemiological surveillance data from the European Centre for Disease Prevention and Control (ECDC).

## Overview

**ECDC** (European Centre for Disease Prevention and Control) collects and provides surveillance data for infectious diseases across Europe.

**Data Coverage:**
- 30 EU/EEA countries
- 50+ infectious diseases
- Historical data from 1990s onwards

**Data Sources:**
- ECDC Surveillance Atlas: https://atlas.ecdc.europa.eu/
- Weekly Threats Reports (CDTR)
- Annual Epidemiological Reports (AER)

**Disease Categories:**
- Vaccine-preventable diseases (Measles, Mumps, Rubella, etc.)
- Respiratory diseases (COVID-19, Influenza, etc.)
- Food and waterborne diseases (Salmonella, Campylobacter, etc.)
- Sexually transmitted infections
- Vector-borne diseases
- Healthcare-associated infections

In [ ]:
# Import necessary libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from epidatasets.sources.ecdc_opendata import ECDCOpenDataAccessor, get_ecdc_diseases, get_ecdc_countries

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

## 1. Initialize ECDC Accessor

In [ ]:
# Initialize the accessor
ecdc = ECDCOpenDataAccessor()

print("✅ ECDC accessor initialized")
print(f"Cache directory: {ecdc.cache_dir}")

## 2. Explore Available Data

### 2.1 List Available Diseases

In [ ]:
# Get list of available diseases
diseases = ecdc.get_available_diseases()

print(f"Total diseases available: {len(diseases)}")
print("\nDiseases by category:")
print(diseases['category'].value_counts())

# Show first 15 diseases
diseases.head(15)

In [ ]:
# Show diseases by category
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

categories = {
    'vaccine_preventable': 'Vaccine-Preventable Diseases',
    'respiratory': 'Respiratory Diseases',
    'foodborne': 'Food & Waterborne Diseases',
    'sti': 'Sexually Transmitted Infections'
}

for idx, (cat, title) in enumerate(categories.items()):
    ax = axes[idx // 2, idx % 2]
    cat_diseases = diseases[diseases['category'] == cat]
    ax.barh(cat_diseases['disease'], range(len(cat_diseases)), color=sns.color_palette("husl", len(cat_diseases)))
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Count')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

### 2.2 List Available Countries

In [ ]:
# Get available countries
countries = ecdc.get_available_countries()

print(f"Total countries covered: {len(countries)}")
print("\nFirst 15 countries:")
countries.head(15)

In [ ]:
# Visualize country distribution
fig, ax = plt.subplots(figsize=(12, 8))

# Sort alphabetically
countries_sorted = countries.sort_values('name')

y_pos = range(len(countries_sorted))
ax.barh(y_pos, [1]*len(countries_sorted), color=sns.color_palette("viridis", len(countries_sorted)))
ax.set_yticks(y_pos)
ax.set_yticklabels(countries_sorted['name'], fontsize=9)
ax.set_xlabel('EU/EEA Member')
ax.set_title('ECDC Surveillance Coverage - EU/EEA Countries', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_xticks([])

plt.tight_layout()
plt.show()

## 3. Disease Surveillance Data

### 3.1 Measles Surveillance in Europe

In [ ]:
# Get measles data for multiple years
measles_data = ecdc.get_disease_data(
    disease="Measles",
    years=range(2019, 2024)
)

print(f"Retrieved {len(measles_data)} records")
print("\nSample data:")
measles_data.head(10)

In [ ]:
# Analyze measles trends by year
measles_by_year = measles_data.groupby('year')['cases'].sum().reset_index()

plt.figure(figsize=(10, 6))
plt.plot(measles_by_year['year'], measles_by_year['cases'], 
         marker='o', linewidth=3, markersize=10, color='#e74c3c')
plt.title('Measles Cases in Europe (2019-2023)', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Total Cases')
plt.grid(True, alpha=0.3)

# Add value labels
for _, row in measles_by_year.iterrows():
    plt.annotate(f"{int(row['cases'])}", 
                xy=(row['year'], row['cases']),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 countries by measles cases in 2023
measles_2023 = measles_data[measles_data['year'] == 2023]
top_countries = measles_2023.nlargest(10, 'cases')

plt.figure(figsize=(12, 6))
plt.barh(top_countries['country'], top_countries['cases'], color='#e67e22')
plt.title('Top 10 Countries - Measles Cases (2023)', fontsize=14, fontweight='bold')
plt.xlabel('Cases')
plt.gca().invert_yaxis()

# Add value labels
for idx, row in top_countries.iterrows():
    plt.text(row['cases'] + 2, row['country'], f"{int(row['cases'])}", 
             va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 3.2 COVID-19 Data Analysis

In [ ]:
# Get COVID-19 data
covid_data = ecdc.get_disease_data(
    disease="COVID-19",
    years=range(2020, 2024)
)

print(f"Retrieved {len(covid_data)} records")

# Show summary by year
covid_summary = covid_data.groupby('year').agg({
    'cases': 'sum',
    'deaths': 'sum'
}).reset_index()

print("\nCOVID-19 Summary by Year:")
print(covid_summary)

In [ ]:
# Plot COVID-19 cases and deaths
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cases
ax1.bar(covid_summary['year'], covid_summary['cases'], color='#3498db', alpha=0.7)
ax1.set_title('COVID-19 Cases in Europe', fontsize=14, fontweight='bold')
ax1.set_xlabel('Year')
ax1.set_ylabel('Total Cases')
ax1.grid(True, alpha=0.3)

# Deaths
ax2.bar(covid_summary['year'], covid_summary['deaths'], color='#e74c3c', alpha=0.7)
ax2.set_title('COVID-19 Deaths in Europe', fontsize=14, fontweight='bold')
ax2.set_xlabel('Year')
ax2.set_ylabel('Total Deaths')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.3 Weekly Surveillance Data

In [ ]:
# Get weekly influenza data for Germany
flu_weekly = ecdc.get_weekly_data(
    disease="Influenza",
    country="DE",
    year=2023
)

print(f"Retrieved {len(flu_weekly)} weeks of data")
print("\nFirst 10 weeks:")
flu_weekly.head(10)

In [ ]:
# Plot weekly influenza data
plt.figure(figsize=(14, 6))

plt.plot(flu_weekly['week_number'], flu_weekly['cases'], 
         marker='o', linewidth=2, markersize=4, color='#2ecc71')
plt.fill_between(flu_weekly['week_number'], flu_weekly['cases'], alpha=0.3, color='#2ecc71')

plt.title('Weekly Influenza Cases - Germany (2023)', fontsize=14, fontweight='bold')
plt.xlabel('Week Number')
plt.ylabel('Cases')
plt.grid(True, alpha=0.3)
plt.xlim(1, 52)

# Add season markers
plt.axvline(x=40, color='orange', linestyle='--', alpha=0.5, label='Flu Season Start')
plt.axvline(x=20, color='blue', linestyle='--', alpha=0.5, label='Flu Season End')
plt.legend()

plt.tight_layout()
plt.show()

## 4. Antimicrobial Resistance (AMR) Data

In [ ]:
# Get AMR data for E. coli and cephalosporins
amr_data = ecdc.get_amr_data(
    pathogen="E. coli",
    antibiotic="cephalosporins",
    year=2023
)

print(f"Retrieved AMR data for {len(amr_data)} countries")
print("\nTop 10 countries by resistance rate:")
amr_data.nlargest(10, 'resistance_percentage')[['country', 'resistance_percentage', 'isolates_tested']]

In [ ]:
# Visualize AMR data
amr_sorted = amr_data.sort_values('resistance_percentage', ascending=True)

plt.figure(figsize=(12, 8))

# Color bars based on resistance level
colors = ['#2ecc71' if x < 20 else '#f39c12' if x < 40 else '#e74c3c' 
          for x in amr_sorted['resistance_percentage']]

plt.barh(amr_sorted['country'], amr_sorted['resistance_percentage'], color=colors)
plt.title('E. coli Resistance to Cephalosporins by Country (2023)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Resistance Percentage (%)')
plt.axvline(x=20, color='green', linestyle='--', alpha=0.5, label='Low <20%')
plt.axvline(x=40, color='orange', linestyle='--', alpha=0.5, label='Moderate 20-40%')

plt.legend()
plt.tight_layout()
plt.show()

## 5. Comparative Analysis

In [ ]:
# Compare multiple diseases
diseases_to_compare = ["Measles", "Mumps", "Rubella"]
comparison_data = []

for disease in diseases_to_compare:
    data = ecdc.get_disease_data(
        disease=disease,
        years=range(2019, 2024)
    )
    yearly = data.groupby('year')['cases'].sum().reset_index()
    yearly['disease'] = disease
    comparison_data.append(yearly)

comparison_df = pd.concat(comparison_data, ignore_index=True)
comparison_pivot = comparison_df.pivot(index='year', columns='disease', values='cases')

print("Vaccine-Preventable Diseases Comparison:")
comparison_pivot

In [ ]:
# Plot comparison
comparison_pivot.plot(kind='bar', figsize=(12, 6), width=0.8)
plt.title('Vaccine-Preventable Diseases in Europe (2019-2023)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Cases')
plt.legend(title='Disease', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Summary Statistics

In [ ]:
# Get summary for 2023
summary = ecdc.get_summary_statistics(year=2023)

print("ECDC Surveillance Summary - 2023")
print("="*60)
for key, value in summary.items():
    if isinstance(value, list):
        print(f"{key.replace('_', ' ').title()}: {', '.join(map(str, value[:5]))}...")
    else:
        print(f"{key.replace('_', ' ').title()}: {value}")

## 7. Export Data

In [ ]:
# Export data to CSV
output_file = "ecdc_measles_surveillance.csv"
measles_data.to_csv(output_file, index=False)

print(f"✅ Data exported to: {output_file}")
print(f"File size: {pd.io.common.file_exists(output_file) and __import__('os').path.getsize(output_file) / 1024:.2f} KB")

# Also export AMR data
amr_file = "ecdc_amr_ecoli.csv"
amr_data.to_csv(amr_file, index=False)
print(f"✅ AMR data exported to: {amr_file}")

## Summary

This notebook demonstrated:

1. ✅ Initializing the ECDC Open Data accessor
2. ✅ Exploring available diseases (50+) and countries (30)
3. ✅ Fetching disease surveillance data (Measles, COVID-19, Influenza)
4. ✅ Analyzing weekly surveillance patterns
5. ✅ Working with antimicrobial resistance data
6. ✅ Comparing multiple diseases across years
7. ✅ Exporting data for further analysis

### Key Features of ECDC Open Data

- **Comprehensive Coverage:** 50+ diseases across 30 EU/EEA countries
- **Historical Data:** Available from 1990s onwards
- **Multiple Formats:** Annual, weekly, and case-based data
- **Standardized:** Comparable across countries and time
- **Open Access:** Free for research and public health use

### Use Cases

- **Disease Surveillance:** Monitor trends in infectious diseases
- **Outbreak Investigation:** Compare patterns across countries
- **Policy Evaluation:** Assess impact of vaccination programs
- **Research:** Epidemiological studies and modeling
- **Public Health Planning:** Resource allocation and preparedness

### Resources

- ECDC Atlas: https://atlas.ecdc.europa.eu/
- Weekly Reports (CDTR): https://www.ecdc.europa.eu/en/publications-data/monitoring/weekly-threats-reports
- Annual Reports: https://www.ecdc.europa.eu/en/publications-data/monitoring/all-annual-epidemiological-reports
- Data Portal: https://www.ecdc.europa.eu/en/publications-data